In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/datasets', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd
import glob
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.utils import resample
import tensorflow as tf
import csv
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout, TextVectorization, GlobalMaxPooling1D, Conv1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
all_files = glob.glob("/content/drive/MyDrive/datasets/*.csv")

df = pd.concat([
    pd.read_csv(f, engine='python', quoting=csv.QUOTE_MINIMAL, on_bad_lines='skip')
    for f in all_files
], ignore_index=True)

print(f"Total rows: {len(df)}")
print(f"Dari {len(all_files)} file: {[os.path.basename(f) for f in all_files]}")
df.info()

Total rows: 18372
Dari 3 file: ['Linkedin_finalfix_1105.csv', 'Jobstreet Cleaned.csv', 'Data_Merged.csv']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18372 entries, 0 to 18371
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                18169 non-null  float64
 1   title             18169 non-null  object 
 2   search_role       18372 non-null  object 
 3   job_description   18337 non-null  object 
 4   extracted_skills  18372 non-null  object 
 5   skills_count      18372 non-null  int64  
 6   search_role_raw   5902 non-null   object 
 7   job_level         5902 non-null   object 
 8   company           15834 non-null  object 
 9   location          15835 non-null  object 
 10  location_raw      5902 non-null   object 
 11  salary_display    5902 non-null   object 
 12  salary_min        1667 non-null   float64
 13  salary_max        1667 non-null   float64
 14  salary_avg        1667 non-n

In [7]:
df = df.drop_duplicates()
df = df.drop(columns=['salary', 'search_role_raw', 'job_level', 'location_raw',
                       'salary_display', 'salary_min', 'salary_max', 'salary_avg', 'company', 'location', 'job_url', 'search_location', 'scraped_at'],
             errors='ignore')
print(f"Total duplicated rows: {df.duplicated().sum()}")
df.dropna(inplace=True)
print(df.isnull().sum())

Total duplicated rows: 388
id                  0
title               0
search_role         0
job_description     0
extracted_skills    0
skills_count        0
dtype: int64


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17998 entries, 0 to 18371
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                17998 non-null  float64
 1   title             17998 non-null  object 
 2   search_role       17998 non-null  object 
 3   job_description   17998 non-null  object 
 4   extracted_skills  17998 non-null  object 
 5   skills_count      17998 non-null  int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 984.3+ KB


In [9]:
def parse_skills(val):
    try:
        result = ast.literal_eval(val)
        return ' '.join(result) if isinstance(result, list) and len(result) > 0 else ''
    except:
        return ''

df['extracted_skills_clean'] = df['extracted_skills'].apply(parse_skills)
print(f"Total rows: {len(df)}")
print(f"Skills kosong: {(df['extracted_skills_clean'] == '').sum()} rows")
print(df['search_role'].value_counts())

Total rows: 17998
Skills kosong: 447 rows
search_role
Backend Developer       2365
Data Engineer           1779
Software Engineer       1478
Data Scientist          1303
Fullstack Developer     1276
DevOps Engineer         1254
Frontend Developer      1104
Data Analyst            1032
Software                1000
Software Developer      1000
Full stack Developer    1000
Developer               1000
IT                       971
Web Developer            831
ML Engineer              381
Progammer                224
Name: count, dtype: int64


In [10]:
min_samples = 900
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)
print(df['search_role'].value_counts())

search_role
Backend Developer       2365
Data Engineer           1779
Software Engineer       1478
Data Scientist          1303
Fullstack Developer     1276
DevOps Engineer         1254
Frontend Developer      1104
Data Analyst            1032
Software Developer      1000
Full stack Developer    1000
Developer               1000
Software                1000
IT                       971
Name: count, dtype: int64


In [11]:
df['search_role'] = df['search_role'].replace({'Full stack Developer': 'Fullstack Developer'})

roles_to_drop = ['Software Engineer', 'Web Developer', 'IT Support', 'IT',
                 'Developer', 'Software', 'System Analyst', 'Progammer']
df = df[~df['search_role'].isin(roles_to_drop)]

min_samples = 200
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)

print(df['search_role'].value_counts())

search_role
Backend Developer      2365
Fullstack Developer    2276
Data Engineer          1779
Data Scientist         1303
DevOps Engineer        1254
Frontend Developer     1104
Data Analyst           1032
Software Developer     1000
Name: count, dtype: int64


In [12]:
encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['search_role'])
num_classes = len(encoder.classes_)
print(f"Total role unik: {num_classes}")
print(f"Mapping kelas:\n{dict(zip(encoder.classes_, range(num_classes)))}")

target_samples = 1200
dfs_resampled = []
for role in df['search_role'].unique():
    df_role = df[df['search_role'] == role]
    if len(df_role) < target_samples:
        df_role = resample(df_role, replace=True, n_samples=target_samples, random_state=42)
    else:
        df_role = df_role.sample(n=target_samples, random_state=42)
    dfs_resampled.append(df_role)

df = pd.concat(dfs_resampled).sample(frac=1, random_state=42).reset_index(drop=True)
df['label'] = encoder.transform(df['search_role'])
print(df['search_role'].value_counts())

df['text_input'] = df['title'] + ' ' + df['job_description'] + ' ' + df['extracted_skills_clean']
df['text_input'] = df['text_input'].fillna('').astype(str)

Total role unik: 8
Mapping kelas:
{'Backend Developer': 0, 'Data Analyst': 1, 'Data Engineer': 2, 'Data Scientist': 3, 'DevOps Engineer': 4, 'Frontend Developer': 5, 'Fullstack Developer': 6, 'Software Developer': 7}
search_role
Data Scientist         1200
Software Developer     1200
DevOps Engineer        1200
Frontend Developer     1200
Backend Developer      1200
Data Engineer          1200
Data Analyst           1200
Fullstack Developer    1200
Name: count, dtype: int64


In [13]:
stop_words_en = set(stopwords.words('english'))
stop_words_id = set(stopwords.words('indonesian'))

all_stop_words = stop_words_en.union(stop_words_id)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)

    words = text.split()
    clean_words = [w for w in words if w not in all_stop_words]

    return ' '.join(clean_words)

df['text_input'] = df['text_input'].apply(clean_text)

In [14]:
print("=== Label alignment check ===")
for i in range(5):
    print(f"Role: {df['search_role'].iloc[i]}")
    print(f"Label: {df['label'].iloc[i]}")
    print(f"Title: {df['title'].iloc[i]}")
    print()

=== Label alignment check ===
Role: Data Scientist
Label: 3
Title: PD Data Analyst

Role: Software Developer
Label: 7
Title: Quality Assurance Engineer

Role: DevOps Engineer
Label: 4
Title: IT Business Partner

Role: Frontend Developer
Label: 5
Title: Fullstack Developer (Node.Js & React.js)

Role: Software Developer
Label: 7
Title: Fullstack Java Engineer



In [15]:
X_train, X_val, y_train, y_val = train_test_split(
    df['text_input'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)

vectorizer = TextVectorization(
    max_tokens=20000,
    output_mode='int',
    output_sequence_length=512,
    ngrams=2
)
vectorizer.adapt(X_train.to_numpy())
print("Shape X_train:", X_train.shape)

Shape X_train: (7680,)


In [16]:
print("=== Sample text inputs ===")
for role in df['search_role'].unique():
    sample = df[df['search_role'] == role]['text_input'].iloc[0]
    print(f"\n[{role}]")
    print(sample[:200])

print("\n=== Vectorizer vocab sample ===")
print(vectorizer.get_vocabulary()[:50])
print(f"\nVocab size: {len(vectorizer.get_vocabulary())}")

=== Sample text inputs ===

[Data Scientist]
pd data analyst aia started exciting movement create healthier sustainable future everyone pioneering innovators years transforming organisation faster simpler connected want even better equipped deve

[Software Developer]
quality assurance engineer job job sourced job board learn job description mencari quality assurance qa memiliki ketelitian berorientasi kualitas fitur aplikasi berjalan optimal posisi berperan penguj

[DevOps Engineer]
business partner deskripsi pekerjaan job description act liaison business users erp data teams gather understand translate business requirements actionable solutions manage stakeholders communicating 

[Frontend Developer]
fullstack developer node js react js looking passionate fullstack software engineer join team build scalable high quality digital products end end responsibilities design develop maintain end end feat

[Backend Developer]
developer lead job developer senior full time updated copy link sha

In [17]:
class CustomEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=5):
        super().__init__()
        self.patience = patience
        self.best_weights = None
        self.best_loss = np.inf
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')
        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n[INFO] val_loss tidak membaik selama {self.patience} epoch. Stop training!")
                self.model.stop_training = True
                print("[INFO] Mengembalikan bobot model ke epoch terbaik.")
                self.model.set_weights(self.best_weights)

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    CustomEarlyStopping(patience=5)
]

In [18]:
vocab_size = len(vectorizer.get_vocabulary())

inputs = Input(shape=(1,), dtype=tf.string, name='text_input')
x = vectorizer(inputs)
x = Embedding(input_dim=vocab_size, output_dim=64)(x)
x = Conv1D(128, 5, activation='relu')(x)
x = GlobalMaxPooling1D()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.5)(x)

outputs = Dense(num_classes, activation='softmax', name='role_output')(x)

model = Model(inputs=inputs, outputs=outputs, name="Job_Role_Classifier")
model.compile(
    optimizer=Adam(learning_rate=0.002),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

print("Mulai proses training...")
history = model.fit(
    X_train.to_numpy(), y_train,
    validation_data=(X_val.to_numpy(), y_val),
    epochs=40,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

Mulai proses training...
Epoch 1/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 0.2251 - loss: 1.9458 - val_accuracy: 0.4115 - val_loss: 1.5883 - learning_rate: 0.0020
Epoch 2/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.4643 - loss: 1.3999 - val_accuracy: 0.5875 - val_loss: 1.1238 - learning_rate: 0.0020
Epoch 3/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.6251 - loss: 1.0113 - val_accuracy: 0.6521 - val_loss: 0.9597 - learning_rate: 0.0020
Epoch 4/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.7228 - loss: 0.7630 - val_accuracy: 0.6802 - val_loss: 0.8747 - learning_rate: 0.0020
Epoch 5/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.7643 - loss: 0.6266 - val_accuracy: 0.6703 - val_loss: 0.8864 - learning_rate: 0.0020
Epoch 6/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.8034 - loss: 0.5301 - val_accuracy: 0.6391 - val_loss: 0.9757 - learning_rate: 0.0020
Epoch 7/40
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - ac

In [19]:
y_pred = np.argmax(model.predict(X_val.to_numpy()), axis=1)
print(classification_report(y_val, y_pred, target_names=encoder.classes_))

60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
                     precision    recall  f1-score   support

  Backend Developer       0.32      0.16      0.21       240
       Data Analyst       0.69      0.88      0.77       240
      Data Engineer       0.79      0.71      0.75       240
     Data Scientist       0.82      0.78      0.80       240
    DevOps Engineer       0.78      0.66      0.71       240
 Frontend Developer       0.66      0.86      0.75       240
Fullstack Developer       0.50      0.53      0.51       240
 Software Developer       0.74      0.87      0.80       240

           accuracy                           0.68      1920
          macro avg       0.66      0.68      0.66      1920
       weighted avg       0.66      0.68      0.66      1920



In [20]:
import joblib
from google.colab import files

model.save('job_role_model.keras')
print("[INFO] Model berhasil disimpan")

joblib.dump(encoder, 'label_encoder.pkl')
print("[INFO] Label encoder berhasil disimpan")

loaded_model = tf.keras.models.load_model('job_role_model.keras')
loaded_encoder = joblib.load('label_encoder.pkl')

def rekomendasi_role(teks_skill_user):
    input_tensor = tf.constant([teks_skill_user], dtype=tf.string)
    pred_probs = loaded_model.predict(input_tensor, verbose=0)
    pred_class_idx = np.argmax(pred_probs)
    predicted_role = loaded_encoder.inverse_transform([pred_class_idx])[0]
    return predicted_role

skill_input = "i have experience for creating RestAPI using Laravel, Fastapi, NestJS, and Golang. I also have experience for developing Database using MySQL, PostgreSQL, and MongoDB "
hasil_prediksi = rekomendasi_role(skill_input)

print("\n--- Hasil Inference ---")
print(f"Skill User : {skill_input}")
print(f"Cocok untuk: {hasil_prediksi}")

[INFO] Model berhasil disimpan
[INFO] Label encoder berhasil disimpan


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- Hasil Inference ---
Skill User : i have experience for creating RestAPI using Laravel, Fastapi, NestJS, and Golang. I also have experience for developing Database using MySQL, PostgreSQL, and MongoDB 
Cocok untuk: DevOps Engineer
